In [ ]:
file_client_path="/Volumes/workspace/default/tp1_datalake/clients.csv"
dfclient= spark.read.format("csv").option("header" , "true").load(file_client_path)
dfclient.printSchema()

In [ ]:
display(dfclient)

In [ ]:
# Vérifier si on a des champs vides
from pyspark.sql import functions as F

null = dfclient.filter(
    F.col("id_client").isNull()
    | (F.trim(F.col("id_client")) == "")
    | F.col("prenom").isNull()
    | (F.trim(F.col("prenom")) == "")
    | F.col("nom").isNull()
    | (F.trim(F.col("nom")) == "")
    | F.col("email").isNull()
    | (F.trim(F.col("email")) == "")
    | F.col("ville").isNull()
    | (F.trim(F.col("ville")) == "")
    | F.col("date_naissance").isNull()
    | (F.trim(F.col("date_naissance")) == "")
    | F.col("date_inscription").isNull()
    | (F.trim(F.col("date_inscription")) == "")
    | F.col("consentement_marketing").isNull()
).count()
display(null)
#(150 champs vides)

In [ ]:
#chercher des ID_client  en double
dfclient.groupBy("id_client") \
    .count() \
    .filter(F.col("count") > 1) \
    .display()

In [ ]:
dfclient.createOrReplaceTempView("dfclient")

In [ ]:
%sql
SELECT id_client,  COUNT(*) AS nombre_occurrences
FROM dfclient
GROUP BY id_client
HAVING COUNT(*) > 1;
-- 0 doublons sur id_client

In [ ]:
%sql
SELECT
    email,
    COUNT(*) AS nombre_occurrences
FROM dfclient
WHERE email is not null
GROUP BY email
HAVING COUNT(*) > 1

-- 46 doublons sur email

In [ ]:
%sql
SELECT
    COUNT(*) AS total_emails_vides
FROM dfclient
WHERE email IS NULL OR TRIM(email) = '';

In [ ]:
#verifier les dates 
dfclient.select(
    "id_client", "date_inscription" , "date_naissance"
).display()
#verifer les dates coherente (ok)
dfclient.filter(
    F.to_date("date_inscription") < F.to_date("date_naissance")
).select(
    "id_client", "date_inscription"
).display()

In [ ]:
#verifer les dates futures  (ok)
dfclient.filter(
    F.to_date("date_inscription") > F.current_date()
).select(
    "id_client", "date_inscription"
).display()

In [ ]:
%sql
SELECT
    consentement_marketing,
    COUNT(*) AS nombre
FROM dfclient
GROUP BY consentement_marketing
ORDER BY consentement_marketing;

In [ ]:
%sql
SELECT COUNT(*) AS nombre_consentements_manquants
FROM dfclient
WHERE consentement_marketing IS NULL
   OR TRIM(consentement_marketing) = '';

In [ ]:
from pyspark.sql import functions as F

# 1. Normaliser les chaînes de caractères
dfclient_silver = dfclient.select(
    F.trim(F.col("id_client")).alias("id_client"),
    F.trim(F.col("prenom")).alias("prenom"),
    F.trim(F.col("nom")).alias("nom"),

    # Normaliser les e-mails : minuscules et espaces supprimés
    F.when(
        F.col("email").isNull() | (F.trim(F.col("email")) == ""),
        F.lit(None).cast("string")
    ).otherwise(
        F.lower(F.trim(F.col("email")))
    ).alias("email"),

    F.trim(F.col("ville")).alias("ville"),

    # Conversion de la date de naissance
    F.to_date(
        F.col("date_naissance"), "yyyy-MM-dd"
    ).alias("date_naissance"),

    # Conversion de la date d'inscription
    F.to_date(
        F.col("date_inscription"), "yyyy-MM-dd"
    ).alias("date_inscription"),

    # Conversion du consentement en booléen
    F.when(
        F.lower(F.trim(F.col("consentement_marketing"))) == "true",
        F.lit(True)
    ).when(
        F.lower(F.trim(F.col("consentement_marketing"))) == "false",
        F.lit(False)
    ).otherwise(
        F.lit(None).cast("boolean")
    ).alias("consentement_marketing")
)

dfclient_silver.printSchema()

# Traitement des anomalies — Table `clients`

## 1. E-mails en double

**Anomalie constatée :** plusieurs clients possèdent la même adresse e-mail.

**Traitement retenu : Dédoublonner, sous condition de vérification.**

**Justification :** une adresse e-mail identique peut indiquer qu'un même client a été enregistré plusieurs fois. Cependant, plusieurs personnes peuvent légitimement partager une adresse e-mail. Il serait donc risqué de supprimer automatiquement des clients uniquement sur la base de leur adresse e-mail.

**Décision :** conserver les enregistrements tant que l'identité des clients n'est pas confirmée. Si plusieurs lignes correspondent réellement au même client, fusionner les fiches en conservant les informations utiles et un identifiant de référence. Sinon, conserver les clients distincts.

## 2. E-mails manquants (`NULL` ou vides)

**Anomalie constatée :** certains clients n'ont pas d'adresse e-mail renseignée.

**Traitement retenu : Isoler.**

**Justification :** aucune autre source de données n'est disponible pour retrouver les adresses manquantes. Il est donc impossible de compléter ces informations de manière fiable. Inventer une adresse ou supprimer les clients entraînerait une perte ou une altération des données.

**Décision :** conserver les clients dans la table principale, maintenir les e-mails manquants à `NULL` et signaler les lignes concernées dans une table de contrôle ou de rejet pour un éventuel enrichissement ultérieur.

## Conclusion

Les doublons d'e-mails sont soumis à une vérification avant tout dédoublonnage, tandis que les e-mails manquants sont isolés pour suivi sans supprimer les clients. Ces décisions permettent de préserver l'intégrité des données et d'éviter toute correction non justifiée.

In [ ]:
# Conserver la table clients nettoyée dans la couche Silver
dfclient_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/tp1_datalake/silver/clients")

In [ ]:
dfclient.createOrReplaceTempView("clients")

In [ ]:
%sql
-- nombre de lignes : 4 999 (5 000 au départ, CLI00042 effacé au TP 4)
SELECT COUNT(*) AS nombre_lignes FROM clients

In [ ]:
%sql
-- recherche de doublons sur id_client (ok)
SELECT id_client, COUNT(*) AS nombre_occurrences
FROM clients
GROUP BY id_client
HAVING COUNT(*) > 1
ORDER BY nombre_occurrences DESC;

In [ ]:
%sql
-- champs vides par colonne : seul email a des vides (150) (ko)
SELECT
  SUM(CASE WHEN id_client IS NULL THEN 1 ELSE 0 END)              AS id_client_vides,
  SUM(CASE WHEN prenom IS NULL THEN 1 ELSE 0 END)                 AS prenom_vides,
  SUM(CASE WHEN nom IS NULL THEN 1 ELSE 0 END)                    AS nom_vides,
  SUM(CASE WHEN email IS NULL OR trim(email) = '' THEN 1 ELSE 0 END) AS email_vides,
  SUM(CASE WHEN ville IS NULL THEN 1 ELSE 0 END)                  AS ville_vides,
  SUM(CASE WHEN date_naissance IS NULL THEN 1 ELSE 0 END)         AS date_naissance_vides,
  SUM(CASE WHEN date_inscription IS NULL THEN 1 ELSE 0 END)       AS date_inscription_vides
FROM clients

In [ ]:
%sql
-- emails partagés par plusieurs clients (45 adresses) (ko : à signaler, cas des foyers)
SELECT email, COUNT(*) AS nombre_clients
FROM clients
WHERE email IS NOT NULL
GROUP BY email
HAVING COUNT(*) > 1
ORDER BY nombre_clients DESC

In [ ]:
%sql
-- format des emails (ok)
SELECT id_client, email
FROM clients
WHERE email IS NOT NULL AND NOT email RLIKE '^[^@ ]+@[^@ ]+[.][^@ ]+$'

In [ ]:
%sql
-- dates incohérentes : inscription avant la naissance ou naissance dans le futur (ok)
SELECT id_client, date_naissance, date_inscription
FROM clients
WHERE to_date(date_inscription) < to_date(date_naissance)
   OR to_date(date_naissance) > current_date()

In [ ]:
%sql
-- clients mineurs au moment de l'inscription (40) (ko : à vérifier avec le marketing, consentement parental)
SELECT id_client, date_naissance, date_inscription,
       floor(months_between(to_date(date_inscription), to_date(date_naissance)) / 12) AS age_inscription
FROM clients
WHERE months_between(to_date(date_inscription), to_date(date_naissance)) / 12 < 18
ORDER BY age_inscription

In [ ]:
%sql
-- valeurs du consentement marketing (ok : true / false uniquement)
SELECT consentement_marketing, COUNT(*) AS nombre
FROM clients
GROUP BY consentement_marketing

### Clients : interpréter, décider, justifier

| Constat | Erreur technique ou réalité métier ? | Décision |
|---|---|---|
| Aucun doublon sur id_client | – | ok, on garde la règle d'unicité (un doublon ferait compter deux fois les ventes) |
| 150 emails vides (3 %) | Réalité métier : l'email est facultatif | **Garder et signaler** (colonne email_manquant). Le client existe et ses achats comptent |
| 45 adresses partagées par 2 clients | Réalité métier possible (couple, famille) | **Garder et signaler** (email_partage) : utile pour le marketing et pour les droits RGPD de Marc |
| 40 clients mineurs à l'inscription | À vérifier : le CRM accepte des moins de 18 ans | **Garder et signaler** au marketing (consentement parental pour le marketing) |
| Dates et formats d'email | Aucune anomalie | ok |

**Pour Sophie** : aucun client n'est supprimé, donc aucune vente ne disparaît du CA.
**Pour Marc** : les données personnelles (nom, prénom, email, date de naissance) restent en silver, et seront minimisées en gold (ville, tranche d'âge).